# CLASSIFICATION
## 3.9. BINARY CLASSIFICATION \
Our problem - predicting churn - is a binary classification problem: \ 
1. the target variable takes one of two values.
2. We encoded it as 1 for customers who churned (the positive class) and 0 for customers who stayed (the negative class).
3. Spam detection is another example: 1 for spam, 0 for not spam.
\
A supervised model is a function that maps features to the target:\
\
g ( x i ) ≈ y i \
 \
For classification, the output of g is a number between 0 and 1: the probability that xi belongs to the positive class. \
If the model outputs 0.8 for a customer, we say it believes there is an 80% chance this customer will churn.\
\
\ ## LINEAR REGRESSION REFRESHER \
g ( x i ) = w0 + w1 X1 + w2 X2 + . . . + wn Xn \
\
## SIGMOID
g ( xi ) = Sigmoid( w0 + w1 x1 + w2 x2 + . . . + wn xn ) \
\
Logistic regression: the same weighted sum as in linear regression, passed through the sigmoid so the output lands between 0 and 1 instead of anywhere on the real line \
\
Sigmoid(z) = 1 / (1 + exp(−z) \
 \
 The sigmoid squashes any real number into the range between 0 and 1. That turns the raw score into a probability. Both models are called linear models, because the score inside is still a linear combination of the features - only the final transformation differs. \
 \
 




In [ ]:
import pandas as pd
import numpy  as np

import matplotlib.pyplot as plt

In [ ]:
# numpy sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# Sanity check on sigmoid function
sigmoid(10000)    # should return 1.0

In [ ]:
# A large positive score maps to 1.0. 
# A large negative score maps to 0, and 
# at exactly 0 the sigmoid gives 0.5 - a coin flip. 
# To see the shape, we evaluate the sigmoid on values from -7 to 7 and plot it

z = np.linspace(-7, 7, 51)
plt.plot(z, sigmoid(z))

# IMPLEMENTING THE SIGMOID MODEL

In [ ]:
# REFRESHER: LINEAR REGRESSION FUNCTION RITTEN WITHOUT A LOOP

def linear_regression(xi):
    result = w0
    
    for j in range(len(w)):
        result = result + xi[j] * w[j]
        
    return result

In [ ]:
# LOGISTIC REGRESSION NEEDS ONLY ONE CHANGE; 
#  PASS THE SCORE THRUGH THE SIGMOID FUNCTION BEFORE RETURNING IT

def logistic_regression(xi):
    score = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result


The two functions are almost identical. \
This is the whole idea of this lesson: \ 
  **Logistic Regression is linear regression with a sigmoid on top**, \
  This which makes the output interpretable as a probability.

### SUMMARY 
**Notes**
 \
In general, supervised models can be represented with this formula: \
 \
g ( xi ) = y i  # "y subscript i" \
 \
Depending on what is the type of target variable, the supervised task can be regression or classification (binary or multiclass). Binary classification tasks can have negative (0) or positive (1) target values. The output of these models is the probability of x i belonging to the positive class. \
 \
Logistic regression is similar to linear regression because both models take into account the bias term and weighted sum of features. The difference between these models is that the output of linear regression is a real number, while logistic regression outputs a value between zero and one, applying the sigmoid function to the linear regression formula. \
 \
g ( xi ) = Sigmoid ( w0 + w1 x 1 + w2x2 + ... + wn xn ) \
    x subscript i \
    w subscript n \
    \
Sigmoid( z ) = 1 / (1 + exp( −z ) \
 \
In this way, the sigmoid function allows transforming a score into a probability.

## 3.10 LOGISTIC REGRESSION WITH Scikit-Learn
Training follows the same pattern as with LinearRegression \
1. create the model
2. fit the model with the feature matrix and target vector

In [ ]:
# STUFF TO RUN BEFORE YOU CAN DO 3.10
#  IMPORT PACKAGES
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
# IMPORT train-test-split FROM PACKAGE sklearn
from sklearn.model_selection import train_test_split

# FROM 3.2 DOWNLOAD AND PREVIEW DATA
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data)

# FROM 3.3. STANDARDIZE COLUMN NAMES
# Preview column names
# print(f"Column Names Before:\r\n {df.columns}")

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# Preview column names
# print()
# print(f"Column Names After:\r\n {df.columns}")

# CONVERT NUMERIC STRINGS TO NUMBERS - PART 1
#   STEP 1: USE THE 'errors='coerce' PARAMETER TO REPLACE CONVERSION ERRORS WITH 'NaN'
tc = pd.to_numeric(df.totalcharges, errors='coerce')

#   STEP 2: COUNT 'NaN' ERRORS
# print(f"tc.isnull().sum()\t{tc.isnull().sum()}")

#   STEP 3: INSPECT 'NaN' records
#    df                              # Full dataframe
#    tc                              # dataframe of indices where the 'tc.isnull()' 
#                                    #  test returns a 'true'
#    ['customerid', 'totalcharges']  # List of columns to return from dataframe 'df'
df[tc.isnull()][['customerid', 'totalcharges']]

# CONVERT NUMERIC STRINGS TO NUMBERS - PART 2
#  SINCE 'totalcharges' REPRESENTS CUSTOMERS WHO JUST JOINED, 
#  THE INITIAL BALANCE SHOULD BE ZERO
#    WE SHOULD REPLACE 'NaN' VALUES WITH ZEROS
#    THIS MAKES SENSE BECAUSE NEW CUSTOMERS HAVE NOT PAID ANYTHING YET
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

# IMPORT train-test-split FROM PACKAGE sklearn
from sklearn.model_selection import train_test_split

# SET ASIDE 20% FOR THE 'df_test' TEST DATASET
#   df_full_train    - name of dataset receive the bulk of the records taken from the full dataset
#   df_test          - name of remainder dataset to receive the remainder of the records from the full dataset
#   train_test_split - method within the 'model_selection' namespace of the 'sklearn' package
#   df               - dataframe containing all records
#   test_size        - percentage of full dataset to place into the remainder dataset
#   random_state     - seed for the random number generator
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# REPEAT THE PROCESS, BUT THIS TIME SPLIT THE 'df_full_train' INTO 'df_train' AND 'df_val'
#  SINCE WE WANT df_val TO CONTAIN 20% OF THE FULL 'df' DATASET, WE WILL NEED TO RECALCULATE
#  THE 'test_size' PARAMETER - 25% OF THE 'df_full_train' DATASET REPRESENTS 20% OF THE FULL 
#  'df' DATASET
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# RESET INDEX VALUES FOR ALL SPLIT DATASETS
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# EXTRACT TARGET / LABEL FROM EACH DATASET
y_train  = df_train.churn.values
y_val    = df_val.churn.values
y_test   = df_test.churn.values

# NOW WE DELETE THE 'churn' COLUMNS FROM FEATURE DATASETS
#  ** DO NOT DELETE BEFORE TARGET / LABEL VECTORS HAVE BEEN CREATED **
del df_train['churn']
del df_val['churn']
del df_test['churn']



In [ ]:
# STUFF TO RUN BEFORE YOU CAN DO 3.10
#  Decide which columns are hold categotical data
#  This can be done with a visual inspection of the data

# Create a list of columns containing numeric values
numerical = [
    'tenure', 
    'monthlycharges', 
    'totalcharges'
] 

# Create a list of columns containing categorical values
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

# check number of unique values in each categorical column
df_full_train[categorical].nunique()

In [ ]:
# STUFF TO RUN BEFORE YOU CAN DO 3.10
# DEBUG
# check number of unique values in each categorical column
df_train[categorical].nunique()

In [ ]:
# STUFF TO RUN BEFORE YOU CAN DO 3.10
# CREATE THE TRAINING DATASET

# Import function used to apply one-hot encoding (OHE)
from sklearn.feature_extraction import DictVectorizer

# Create the 'dv' matrix 
#   a 'sparse' matrix is a scipy matrix one where most of the values are zeros
dv = DictVectorizer(sparse=False) 

# Create the training dictionary
#  NOTE: If you do not set "orient='records'", the 'to_dict()' method will
#        turn each coumn into a dictionary and we want each row to become
#        a dictionary
train_dict = df_train[categorical + numerical].to_dict(orient='records')

# Create training dataset with one-hot encoded columns
#  NOTE: the 'fit_transform()' method is equivalent to 
#          dv      = DictVectorizer(sparse=false)
#          X_train = dv.fit(train_dicts)     # first 'fit' the data
#          X_train = dv.transform(X_train)   # then 'transform' the data
X_train = dv.fit_transform(train_dict)

In [ ]:
# STUFF TO RUN BEFORE YOU CAN DO 3.10
#   ENCODE THE VALIDATION DATASET
val_dict = df_val[categorical + numerical].to_dict(orient='records')
#  NOTE: We do not want to 'fit' the validation dataset
#        this is wny we only 'transform' the dataset
#  NOTE: When building the 
X_val = dv.transform(val_dict)

In [ ]:
# DEBUG
print(X_train.shape, y_train.shape)

In [ ]:
# DEBUG
# # import numpy as np
# X_train
# y_train

#np.where(y_train == "Yes", 1, 0)

y_train = np.where(y_train == "Yes", True, False)
# y_train

In [ ]:
y_train

In [ ]:
# DEBUG
# import numpy as np

unique_values = np.unique(y_train)
print(unique_values.tolist())


### 'sklearn' 'LogisticRegression' VOCABULARY
  intercept    synonym of 'bias' 
### SYNTAX
  from sklearn.linear_model import LogisticRegression \
\
  model = LogisticRegression(solver='lbfgs') \
  model = LogisticRegression() \
  model.fit(X_train, y_train) \
  model.intercept_[0]      # calculate predictions \
  model.coef_[0]           # feature weights (referred to as 'w' in prior lessons) \
  model.coef_[0].round(3)  # round feature weights to 3 digits of precision 
  

In [ ]:
# DEBUG
# from sklearn.linear_model import LogisticRegression

# model = LogisticRegression(solver='lbfgs', max_iter=10000)

In [ ]:
# DEBUG
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
#
# model = make_pipeline(
#     StandardScaler(),
#     LogisticRegression(max_iter=3000)
# )
# model.fit(X_train, y_train)

In [ ]:
# DEBUG
# model.coef_[0].round(3)

In [ ]:
# DEBUG
# model.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='lbfgs' , max_iter=10000)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly

# 'fit() method is used to train the model
model.fit(X_train, y_train)

In [ ]:
# DEBUG - VALUE IS WRONG
model.intercept_[0]

#  LESSON SAYS RETURN SHOULD BE -0.10903395348323511

In [ ]:
# DEBUG - VALUES ARE ALL WRONG
model.coef_[0].round(3)

#  LESSON SAYS RETURN SHOULD BE
# array([ 0.475, -0.175, -0.408, -0.03 , -0.078,  0.063, -0.089, -0.081,
#       -0.034, -0.073, -0.335,  0.316, -0.089,  0.004, -0.258,  0.141,
#        0.009,  0.063, -0.089, -0.081,  0.266, -0.089, -0.284, -0.231,
#        0.124, -0.166,  0.058, -0.087, -0.032,  0.07 , -0.059,  0.141,
#       -0.249,  0.215, -0.12 , -0.089,  0.102, -0.071, -0.089,  0.052,
#        0.213, -0.089, -0.232, -0.07 ,  0.   ])

In [ ]:
# MAKING PREDICTIONS
#  NOTE: 'predict_proba' returns a matrix with 2 columns
#           col 1   probability of class 0
#           col 2   probability of class 1
#         sum of the two columns always adds up to 1.0

# model.predict_proba(X_val)[:, 1]
y_pred = model.predict_proba(X_val)[:, 1]

In [ ]:
# DEBUG
# y_pred

In [ ]:
# IF PROBABILITY OF CHURN IS 50% OR MORE

# y_pred >= 0.5
np.where(y_pred == False, 0, 1)
# y_pred = np.where(y_pred == "False", 0, 1)
# churn_decision = (y_pred >= 0.5)

In [ ]:
# CHECK FOR ACCURACY #1
(y_val == churn_decision).mean()

In [ ]:
# CHECK FOR ACCURACY #2
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = churn_decision.astype(int)
df_pred['actual'] = y_val
df_pred

In [ ]:
# MARK ROWS WHERE PREDICTION MATCHES REALITY
df_pred['correct'] = df_pred.prediction == df_pred.actual
df_pred.correct.mean()

In [ ]:
# DEBUG
df_pred.correct